# MorphCell RBC Feature Extraction

## 1. Setup

### 1.1 Environment Setup and Imports

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent.parent.parent
sys.path.insert(0, str(project_root))
print("Project root:", project_root)

Project root: /home/verve/Project/research/MorphCell


In [2]:
import torch
import numpy as np
from tqdm.auto import tqdm

from morphcell.data import HDF5Dataset
from morphcell import MorphCellInference

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/home/verve/Project/research/MorphCell/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.7.1+cu118
CUDA available: True
Using device: cuda


### 1.2 Configuration

In [3]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

BATCH_SIZE = 32
CONFIG_DIR = project_root / "morphcell/config"

# Cross+self
CONFIG_NAME = "system/morphcell_cross+self"
CHECKPOINT_PATH = project_root / "checkpoints/morphcell_cross_self.ckpt"
OUTPUT_DIR = project_root / "outputs/experiments/rbc/features/morphcell_cross+self"

# Direct+self
# CONFIG_NAME = "system/morphcell_direct+self"
# CHECKPOINT_PATH = project_root / "checkpoints/morphcell_direct_self.ckpt"
# OUTPUT_DIR = (
#     project_root / "outputs/experiments/rbc/features/morphcell_direct+self"
# )

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
print(f"Config dir: {CONFIG_DIR}")
print(f"Config name: {CONFIG_NAME}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

Config dir: /home/verve/Project/research/MorphCell/morphcell/config
Config name: system/morphcell_cross+self
Checkpoint path: /home/verve/Project/research/MorphCell/checkpoints/morphcell_cross_self.ckpt
Output dir: /home/verve/Project/research/MorphCell/outputs/experiments/rbc/features/morphcell_cross+self


## 2. Model and Data Preparation
### 2.1 Load Pretrained Model

In [4]:
inferencer = MorphCellInference(
    config_dir=str(CONFIG_DIR),
    config_name=CONFIG_NAME,
    checkpoint_path=str(CHECKPOINT_PATH),
    device="auto",
    batch_size=BATCH_SIZE,
)

print("✓ Model loaded successfully!")
print(f"Model is on: {inferencer.get_device()}")
print(f"Capabilities: {inferencer.get_capabilities()}")

✓ Model loaded successfully!
Model is on: cuda
Capabilities: ModelCapabilities(global_features=True, local_features=True, reconstruct_from_input=True, reconstruct_from_features=True, feature_fusion=True, cross_reconstruction=True, saliency=True)


### 2.2 Load RBC Dataset

In [5]:
dataset = HDF5Dataset(
    path=f"{project_root}/datasets/redblood",
    splits=["set1"],
    normalize=True,
    normalize_scale=None,
    pca_align=True,
)

print(f"Dataset loaded: {len(dataset)} samples")
print(f"Available classes ({len(dataset.sorted_names)}): {dataset.sorted_names}")

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True if device.type == "cuda" else False,
)

Dataset loaded: 620 samples
Available classes (7): ['discocyte', 'echinocyte1', 'echinocyte2', 'echinocyte3', 'stomatocyte', 'spherocyte', 'knizocyte']


### 2.3 Extract Features and Save Arrays

In [6]:
all_cls_features = []
all_patch_features = []
all_concated_features = []
all_ids = []
all_names = []
all_labels = []
all_scale_factors = []

print("Extracting MorphCell features...")
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Extracting features"):
        points = batch["points"]
        feature_bundle = inferencer.extract_features(points, normalize=True)

        all_cls_features.append(feature_bundle.global_features.cpu().numpy())
        all_patch_features.append(feature_bundle.local_features.cpu().numpy())
        all_concated_features.append(feature_bundle.concated_features.cpu().numpy())
        all_ids.extend(batch["id"])
        all_names.extend(batch["name"])
        all_labels.append(batch["label"].cpu().numpy())
        all_scale_factors.append(batch["scale_factor"].cpu().numpy())

cls_features = np.concatenate(all_cls_features, axis=0)
patch_features = np.concatenate(all_patch_features, axis=0)
concated_features = np.concatenate(all_concated_features, axis=0)
ids = np.array(all_ids)
names = np.array(all_names)
labels = np.concatenate(all_labels, axis=0).reshape(-1)
scales = 1 / np.concatenate(all_scale_factors, axis=0).reshape(-1)

np.save(OUTPUT_DIR / "ids.npy", ids)
np.save(OUTPUT_DIR / "names.npy", names)
np.save(OUTPUT_DIR / "labels.npy", labels)
np.save(OUTPUT_DIR / "scales.npy", scales)
np.save(OUTPUT_DIR / "cls_features.npy", cls_features)
np.save(OUTPUT_DIR / "patch_features.npy", patch_features)
np.save(OUTPUT_DIR / "concated_features.npy", concated_features)

print("\n✓ Extraction completed!")
print(f"CLS features shape: {cls_features.shape}")
print(f"Patch features shape: {patch_features.shape}")
print(f"Concatenated features shape: {concated_features.shape}")
print(f"IDs shape: {ids.shape}")
print(f"Names shape: {names.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Scales shape: {scales.shape}")
print(f"Unique labels: {np.unique(labels)}")

Extracting MorphCell features...


Extracting features: 100%|██████████| 20/20 [00:02<00:00,  8.13it/s]


✓ Extraction completed!
CLS features shape: (620, 384)
Patch features shape: (620, 64, 384)
Concatenated features shape: (620, 768)
IDs shape: (620,)
Names shape: (620,)
Labels shape: (620,)
Scales shape: (620,)
Unique labels: [0 1 2 3 4 5 6]
